In [ ]:
input_raster = r"C:\Users\KyleSteen\Documents\ArcGIS\Projects\Exclusion_Analysis_started_on_2_17_2026\Exclusion_Rasters\conus_composite_solar_exclusion_10m_5070.tif"
mask_shp = r"C:\Users\KyleSteen\Documents\Final_ROWs_CONUS_10m_Buffer.shp"
output_raster = r"C:\Users\KyleSteen\Documents\Buffer_10m_ExtractbyMask.tif"

In [ ]:
import rasterio
from rasterio.windows import from_bounds, Window
from rasterio.features import geometry_mask
import geopandas as gpd
import numpy as np
import os
import logging
import time

# -----------------------------
# Logging setup
# -----------------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")

# -----------------------------
# File paths
# -----------------------------
input_raster = r"C:\Users\KyleSteen\Documents\ArcGIS\Projects\Exclusion_Analysis_started_on_2_17_2026\Exclusion_Rasters\conus_composite_solar_exclusion_10m_5070.tif"
mask_shp = r"C:\Users\KyleSteen\Documents\Final_ROWs_CONUS_10m_Buffer.shp"
output_raster = r"C:\Users\KyleSteen\Documents\Buffer_10m_ExtractbyMask.tif"

output_folder = r'C:\Users\KyleSteen\Documents'

os.makedirs(output_folder, exist_ok=True)

# -----------------------------
# Parameters
# -----------------------------
tile_size = 50000  # pixels (50k x 50k)
nodata_value = -9999  # adjust to your raster

# -----------------------------
# Start timer
# -----------------------------
start_time = time.time()

logging.info("Reading AOI shapefile...")
gdf = gpd.read_file(mask_shp)

with rasterio.open(input_raster) as src:
    if gdf.crs != src.crs:
        gdf = gdf.to_crs(src.crs)

    # Full AOI bounds
    bounds = gdf.total_bounds
    full_window = from_bounds(*bounds, transform=src.transform)
    full_window = full_window.round_offsets().round_lengths()
    logging.info(f"Full AOI window: {full_window}")

    # Compute number of tiles in x/y
    n_tiles_x = int(np.ceil(full_window.width / tile_size))
    n_tiles_y = int(np.ceil(full_window.height / tile_size))
    total_tiles = n_tiles_x * n_tiles_y
    completed_tiles = 0

    logging.info(f"Processing {total_tiles} tiles ({n_tiles_x} x {n_tiles_y})")

    # Loop over tiles
    for ty in range(n_tiles_y):
        for tx in range(n_tiles_x):

            row_off = ty * tile_size
            col_off = tx * tile_size

            w_height = min(tile_size, full_window.height - row_off)
            w_width = min(tile_size, full_window.width - col_off)

            window = Window(
                col_off + full_window.col_off,
                row_off + full_window.row_off,
                w_width,
                w_height
            )

            # Read tile
            data = src.read(window=window)

            # Mask for this tile
            transform = src.window_transform(window)
            mask_array = geometry_mask(
                gdf.geometry,
                transform=transform,
                invert=True,
                out_shape=(w_height, w_width)
            )

            for band in range(data.shape[0]):
                data[band][~mask_array] = nodata_value

            # Output metadata
            out_meta = src.meta.copy()
            out_meta.update({
                "height": w_height,
                "width": w_width,
                "transform": transform,
                "compress": "lzw",
                "tiled": True,
                "BIGTIFF": "YES",
                "nodata": nodata_value
            })

            # Write tile
            out_file = os.path.join(output_folder, f"tile_{ty}_{tx}.tif")
            with rasterio.open(out_file, "w", **out_meta) as dst:
                dst.write(data)

            # Update progress
            completed_tiles += 1
            elapsed = time.time() - start_time
            remaining_tiles = total_tiles - completed_tiles
            percent_done = (completed_tiles / total_tiles) * 100
            eta = elapsed * (total_tiles / completed_tiles - 1) if completed_tiles > 0 else 0

            logging.info(f"Completed tile {completed_tiles}/{total_tiles} | "
                         f"{percent_done:.2f}% | Remaining tiles: {remaining_tiles} | "
                         f"ETA: {eta/60:.2f} min")


In [ ]:
import rasterio
from rasterio.windows import from_bounds, Window
from rasterio.features import geometry_mask
import geopandas as gpd
import numpy as np
import os
import logging
import time
from shapely.geometry import mapping

# -----------------------------
# Logging setup
# -----------------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")

# -----------------------------
# File paths
input_raster = r"C:\Users\KyleSteen\Documents\ArcGIS\Projects\Exclusion_Analysis_started_on_2_17_2026\Exclusion_Rasters\conus_composite_solar_exclusion_10m_5070.tif"
mask_shp = r"C:\Users\KyleSteen\Documents\Final_ROWs_CONUS_10m_Buffer.shp"

output_folder = r"C:\Users\KyleSteen\Documents\output_tiles"

os.makedirs(output_folder, exist_ok=True)

# -----------------------------
# Parameters
# -----------------------------
tile_size = 20000  # 20k x 20k pixels
nodata_value = -9999

# -----------------------------
# Read AOI shapefile and serialize geometry
# -----------------------------
start_time = time.time()
logging.info("Reading AOI shapefile...")
gdf = gpd.read_file(mask_shp)
serialized_geom = [mapping(geom) for geom in gdf.geometry]

# -----------------------------
# Open raster and compute tiles
# -----------------------------
with rasterio.open(input_raster) as src:
    if gdf.crs != src.crs:
        raise ValueError("AOI CRS does not match raster CRS. Please reproject.")

    bounds = gdf.total_bounds
    full_window = from_bounds(*bounds, transform=src.transform)
    full_window = full_window.round_offsets().round_lengths()
    logging.info(f"Full AOI window: {full_window}")

    n_tiles_x = int(np.ceil(full_window.width / tile_size))
    n_tiles_y = int(np.ceil(full_window.height / tile_size))
    total_tiles = n_tiles_x * n_tiles_y
    logging.info(f"Processing {total_tiles} tiles ({n_tiles_x} x {n_tiles_y}) sequentially")

    # Prepare tiles
    tiles = []
    for ty in range(n_tiles_y):
        for tx in range(n_tiles_x):
            row_off = ty * tile_size
            col_off = tx * tile_size
            w_height = min(tile_size, full_window.height - row_off)
            w_width = min(tile_size, full_window.width - col_off)
            window = Window(
                col_off + full_window.col_off,
                row_off + full_window.row_off,
                w_width,
                w_height
            )
            tiles.append((ty, tx, window))

# -----------------------------
# Sequential tile processing
# -----------------------------
completed_tiles = 0

for ty, tx, window in tiles:
    tile_start = time.time()
    logging.info(f"Starting tile ({ty},{tx}) {completed_tiles+1}/{total_tiles}")

    with rasterio.open(input_raster) as src:
        # Read tile
        data = src.read(window=window)
        transform = src.window_transform(window)

        # Mask for this tile
        mask_array = geometry_mask(
            serialized_geom,
            transform=transform,
            invert=True,
            out_shape=(window.height, window.width)
        )
        for band in range(data.shape[0]):
            data[band][~mask_array] = nodata_value

        # Output metadata
        out_meta = src.meta.copy()
        out_meta.update({
            "height": window.height,
            "width": window.width,
            "transform": transform,
            "compress": "lzw",
            "tiled": True,
            "BIGTIFF": "YES",
            "nodata": nodata_value
        })

        # Write tile
        out_file = os.path.join(output_folder, f"tile_{ty}_{tx}.tif")
        with rasterio.open(out_file, "w", **out_meta) as dst:
            dst.write(data)

    completed_tiles += 1
    elapsed = time.time() - start_time
    tile_elapsed = time.time() - tile_start
    remaining_tiles = total_tiles - completed_tiles
    percent_done = (completed_tiles / total_tiles) * 100
    eta = elapsed * (total_tiles / completed_tiles - 1) if completed_tiles > 0 else 0
    mb_processed = data.nbytes / (1024**2)

    logging.info(f"Completed tile ({ty},{tx}) {completed_tiles}/{total_tiles} | "
                 f"{percent_done:.2f}% | Remaining: {remaining_tiles} | "
                 f"ETA: {eta/60:.2f} min | Tile size: {mb_processed:.2f} MB | "
                 f"Tile time: {tile_elapsed/60:.2f} min")


2026-02-18 13:38:09,993 - Reading AOI shapefile...
2026-02-18 13:38:29,000 - Full AOI window: Window(col_off=59379, row_off=16782, width=440481, height=277455)
2026-02-18 13:38:29,000 - Processing 322 tiles (23 x 14) sequentially
2026-02-18 13:38:29,006 - Starting tile (0,0) 1/322
